## 04: Data Quality Assessment

### Introduction

This notebook implements the **Data Understanding / Evaluation** phase of the CRISP-DM workflow
for the Divvy bike-share project. It picks up where `02_database_design.ipynb` and
`03_etl_pipeline.ipynb` leave off, and asks a simple but critical question: now that millions of
trip records have been loaded into the `divvy` schema (`dim_date`, `dim_station`, `dim_ride_type`,
`dim_member_type`, `fact_trip`) inside `divvy_db`, can that data actually be trusted?

**The pipeline:**

1. **Connects** directly to the live `divvy_db` warehouse and pulls every table into memory once, so
   every check that follows works from one consistent snapshot instead of re-querying the database
   repeatedly.
2. **Runs nine independent quality checks** — duplicate IDs, missing stations, invalid timestamps,
   negative durations, impossible coordinates, null rates, cardinality, completeness, and
   consistency — each re-derived straight from the loaded data rather than trusted from the ETL
   pipeline's own self-reported counters.
3. **Visualizes missingness** two ways: a table x column heatmap of null percentages, and a
   row-level presence matrix that reveals whether missing values cluster together.
4. **Rolls every check up** into a single weighted 0–100 Data Quality Score across five
   industry-standard dimensions — Uniqueness, Completeness, Validity, Consistency, and Accuracy —
   with a letter grade.
5. **Publishes a structured report** in three formats (Markdown, JSON, CSV), so the findings can be
   read by a person, consumed by another script, or dropped straight into a BI tool.

**Why this matters:** loading data into a warehouse is only half the job —
knowing whether the data you loaded can actually be trusted is the other half. This notebook
demonstrates that discipline in practice: it never assumes the pipeline worked correctly, it
re-verifies every guarantee (uniqueness, referential integrity, valid ranges) directly against the
warehouse, distinguishes genuine errors from expected edge cases (e.g. a valid trip that ends outside
Chicago vs. a physically impossible coordinate), and turns raw findings into a single, defensible
score that a non-technical stakeholder can track over time. That is the same audit mindset used to
vet production data pipelines and machine learning training data before anyone builds a model or a
dashboard on top of them.

**Skills demonstrated:** independent data-quality auditing, SQL/warehouse validation,
referential-integrity testing, statistical outlier detection, missing-data analysis, composite
scoring/KPI design, automated multi-format reporting, and translating technical findings into a
business-readable deliverable.

#### Checks performed
1. Duplicate ride IDs
2. Missing stations
3. Invalid timestamps
4. Negative ride durations
5. Impossible coordinates
6. Null percentages
7. Cardinality
8. Data completeness
9. Data consistency

#### Deliverables
- **Data Quality Report** — a structured, human-readable Markdown/JSON/CSV report with PASS/WARN/FAIL
  verdicts for every check
- **Missing value matrix** — a table-level null-percentage heatmap and a row-level presence matrix
- **Quality score** — a single weighted 0–100 composite score (with letter grade) across five
  data-quality dimensions: Uniqueness, Completeness, Validity, Consistency, and Accuracy

### Import Libraries

Brings in everything the audit needs: `pandas`/`numpy` for the data manipulation and statistics
behind every check, `psycopg2` to talk to the PostgreSQL warehouse built in `02_database_design.ipynb`,
`matplotlib` for the two chart types produced later, and `IPython.display` to render the final report
as formatted Markdown directly in the notebook. The local `config` module keeps database credentials
in an external `database.ini` file rather than hardcoded in the notebook — the same secure-credentials
pattern used in every other notebook in this project. A single `warnings.filterwarnings` call quiets a
harmless `psycopg2`/pandas compatibility notice so the rest of the output stays focused on the actual
findings.

**Skills demonstrated:** secure credential handling, consistent tooling and conventions carried across
a multi-notebook pipeline.

In [1]:
import json
import os
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import config

warnings.filterwarnings("ignore", category=UserWarning)  # psycopg2 + pandas read_sql notice
plt.rcParams["figure.facecolor"] = "white"


### Configuration

Centralizes every constant this audit depends on, so nothing is a "magic number" buried inside a
check: where the report files get written, how many rows to sample while iterating on the notebook
(`SAMPLE_ROWS`, left `None` for a full assessment), the geographic bounding boxes used to separate
genuine Chicago-area trips from physically impossible coordinates, Divvy's actual program launch date
(used to flag any timestamp that predates the service or sits in the future), a small tolerance for
comparing a stored duration against a recomputed one, and the two percentage thresholds
(`FAIL_THRESHOLD_PCT`, `WARN_THRESHOLD_PCT`) that turn a raw "% of rows affected" number into a
PASS/WARN/FAIL verdict for every check that follows.

Defining severity thresholds once, up front, rather than eyeballing each result individually, is what
makes every check below consistent and auditable — anyone reviewing this notebook can see exactly what
"FAIL" means before looking at a single result.

**Skills demonstrated:** configuration-driven design, explicit and auditable business-rule thresholds,
reproducible analysis setup.

In [2]:
# Project folder (comprehensive_divvy_bike_sharing_analytics), one level up from notebooks/
PROJECT_FOLDER = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
REPORTS_FOLDER = os.path.join(PROJECT_FOLDER, "reports", "data_quality_reports")
os.makedirs(REPORTS_FOLDER, exist_ok=True)

SCHEMA = "divvy"

# Set an integer to LIMIT the fact_trip pull while iterating on this notebook quickly
# (e.g. 8_000_000); leave as None to assess the full table before finalizing the report.
#SAMPLE_ROWS = None
SAMPLE_ROWS = 8_000_000

# Loose Chicagoland bounding box (mirrors 03_data_cleaning_and_loading.ipynb) -- used to flag
# geospatial *outliers* (out of service area), separate from physically IMPOSSIBLE coordinates.
CHI_LAT_RANGE = (41.55, 42.15)
CHI_LNG_RANGE = (-88.10, -87.40)

# Physically valid ranges for any coordinate on Earth
WORLD_LAT_RANGE = (-90.0, 90.0)
WORLD_LNG_RANGE = (-180.0, 180.0)

# Divvy's program launch date -- timestamps before this (or in the future) are suspect
PROGRAM_LAUNCH = pd.Timestamp("2013-06-01", tz="UTC")

# A stored duration_minutes that disagrees with (ended_at - started_at) by more than this many
# minutes indicates an ETL/derivation bug rather than rounding noise
DURATION_TOLERANCE_MIN = 0.5

# Severity thresholds applied to the *share of rows affected* by each check
FAIL_THRESHOLD_PCT = 1.0   # >=1% of rows affected -> FAIL
WARN_THRESHOLD_PCT = 0.1   # >=0.1% of rows affected -> WARN, else PASS

RUN_TIMESTAMP = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Reports will be written to: {REPORTS_FOLDER}")
print(f"Run timestamp (UTC): {RUN_TIMESTAMP}")


Reports will be written to: C:\Projects\comprehensive_divvy_bike_sharing_analytics\reports\data_quality_reports
Run timestamp (UTC): 2026-09-13T13:46:26+00:00


### Database Connection

Three small helper functions that the rest of the notebook builds on. `connect_divvy()` opens a
connection to the warehouse using the same credential pattern as `02_database_design.ipynb` and
`03_etl_pipeline.ipynb`, catching any connection error and returning `None` instead of crashing, so the
caller can fail fast with a clear message. `run_query()` wraps `pandas.read_sql_query` so every check
below can pull data with one line of code. `verdict()` is the single source of truth for turning a
"% of rows affected" number into a PASS, WARN, or FAIL label using the thresholds set in Configuration —
meaning every one of the nine checks in this notebook grades itself the exact same way.

**Skills demonstrated:** reusable helper-function design, defensive error handling, single-responsibility
functions, consistent grading logic applied uniformly across independent checks.

In [3]:
def connect_divvy():
    """Connect to the divvy_db database (mirrors 02_database_design.ipynb)."""
    conn = None
    try:
        params = config.config_divvy()
        conn = psycopg2.connect(**params)
        conn.autocommit = True
        return conn
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)
        return None


def run_query(conn, sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)


def verdict(pct_affected: float) -> str:
    """Map a percentage of affected rows to a PASS / WARN / FAIL verdict."""
    if pct_affected >= FAIL_THRESHOLD_PCT:
        return "FAIL"
    if pct_affected >= WARN_THRESHOLD_PCT:
        return "WARN"
    return "PASS"


### Load Data for Assessment

Pulls every table in the `divvy` schema into memory once, so every check below reuses the same
snapshot — a consistent, point-in-time assessment — instead of re-querying the live database for
every single check.

Connects to `divvy_db`, pulls the full `fact_trip` table (ordered by its surrogate key, optionally
capped by `SAMPLE_ROWS` while iterating) plus all four dimension tables, then immediately closes the
connection — from that point on, the notebook does its analysis entirely in pandas, so the warehouse
is never left holding an open connection while nine separate checks run. Timestamp columns are
explicitly recast to timezone-aware `datetime64`, since `psycopg2` returns native Python `datetime`
objects that pandas doesn't automatically treat as a proper time series. A one-line row/column count
printout for every table is a quick sanity check that data actually loaded before any check runs.

**Skills demonstrated:** efficient single-pull data access patterns, connection lifecycle management,
dtype correctness for time-series analysis.

In [4]:
conn = connect_divvy()
if conn is None:
    raise ConnectionError("Could not connect to divvy_db.")

_limit_clause = f"LIMIT {SAMPLE_ROWS}" if SAMPLE_ROWS else ""

fact = run_query(conn, f"SELECT * FROM {SCHEMA}.fact_trip ORDER BY trip_key {_limit_clause};")
dim_date = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_date;")
dim_station = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_station;")
dim_ride_type = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_ride_type;")
dim_member_type = run_query(conn, f"SELECT * FROM {SCHEMA}.dim_member_type;")

conn.close()

# Force proper datetime64 dtype -- psycopg2 returns native datetime objects
fact["started_at"] = pd.to_datetime(fact["started_at"], utc=True)
fact["ended_at"] = pd.to_datetime(fact["ended_at"], utc=True)
dim_date["full_date"] = pd.to_datetime(dim_date["full_date"])

tables = {
    "fact_trip": fact,
    "dim_date": dim_date,
    "dim_station": dim_station,
    "dim_ride_type": dim_ride_type,
    "dim_member_type": dim_member_type,
}

for name, tdf in tables.items():
    print(f"{name:<16} rows={len(tdf):>10,}  cols={tdf.shape[1]}")


fact_trip        rows= 8,000,000  cols=20
dim_date         rows=       974  cols=10
dim_station      rows=     3,943  cols=6
dim_ride_type    rows=         3  cols=2
dim_member_type  rows=         2  cols=2


### Data Quality Checks

Every check below records its metrics into one shared `findings` dictionary — keyed by check name —
which is what drives the final report, the missing-value matrix, and the composite quality score
built later in the notebook. Starting from an empty dict here means each of the nine checks that
follow is self-contained and independently re-runnable, while still contributing to the same final
output.

**Skills demonstrated:** shared-state design that keeps otherwise-independent checks composable into
one unified report.

In [5]:
findings = {}  # check_name -> dict of metrics, populated by every section below


#### Duplicate Ride IDs

`ride_id` is the fact table's natural key for every trip and is declared `UNIQUE` at the database
level in `02_database_design.ipynb`. This check doesn't just trust that constraint — it independently
re-verifies uniqueness directly against the data that was actually loaded, because a database
constraint can be bypassed (e.g. by a bulk `COPY`, a manual edit, or a schema created without it).

Counts how many times each `ride_id` appears, isolates any that show up more than once, and reports
both the number of duplicate groups and the total rows involved as a percentage of the whole fact
table — then displays the first ten duplicate rows so an analyst can see exactly which trips are
affected, not just how many.

**Skills demonstrated:** independent constraint verification, primary-key/uniqueness auditing,
percentage-based severity scoring.

In [6]:
ride_id_counts = fact["ride_id"].value_counts()
dup_groups = ride_id_counts[ride_id_counts > 1]
dup_rows = fact[fact["ride_id"].isin(dup_groups.index)]

n_dup_rows = len(dup_rows)
pct_dup_rows = round(100 * n_dup_rows / len(fact), 4) if len(fact) else 0.0

findings["duplicate_ride_ids"] = {
    "duplicate_ride_id_groups": int(len(dup_groups)),
    "duplicate_rows_total": int(n_dup_rows),
    "pct_rows_affected": pct_dup_rows,
    "verdict": verdict(pct_dup_rows),
}

print(f"Duplicate ride_id groups : {len(dup_groups):,}")
print(f"Rows involved in duplicates: {n_dup_rows:,} ({pct_dup_rows}% of fact_trip)")
display(dup_rows.sort_values('ride_id').head(10)[["trip_key", "ride_id", "started_at", "ended_at"]])


Duplicate ride_id groups : 0
Rows involved in duplicates: 0 (0.0% of fact_trip)


,trip_key,ride_id,started_at,ended_at


#### Missing Stations

Station data can go wrong in three distinct ways, and this check tests for each of them separately
rather than lumping them into one number: (a) trips where the start or end station key is simply
`NULL`, (b) "orphaned" foreign keys — a station key recorded on a trip that doesn't correspond to any
real row in `dim_station` (a referential-integrity failure that should be impossible given the
foreign-key constraints from `02_database_design.ipynb`, but is worth re-checking directly), and
(c) station dimension rows that exist but are themselves incomplete, missing a name or coordinates.

Every one of those failure modes gets its own count, and the row-level ones (NULL start/end keys) are
rolled into a single "% of rows affected" verdict, since those are the ones that actually block
trip-level analysis.

**Skills demonstrated:** decomposing a single "is it broken" question into multiple, independently
diagnosable root causes; referential-integrity testing across a foreign-key relationship.

In [7]:
null_start_station = fact["start_station_key"].isna().sum()
null_end_station = fact["end_station_key"].isna().sum()

valid_station_keys = set(dim_station["station_key"])
orphan_start = fact.loc[fact["start_station_key"].notna() & ~fact["start_station_key"].isin(valid_station_keys)]
orphan_end = fact.loc[fact["end_station_key"].notna() & ~fact["end_station_key"].isin(valid_station_keys)]

station_missing_name = dim_station["station_name"].isna().sum()
station_missing_coords = dim_station[["latitude", "longitude"]].isna().any(axis=1).sum()

total_missing_station_rows = int(((fact["start_station_key"].isna()) | (fact["end_station_key"].isna())).sum())
pct_missing_station = round(100 * total_missing_station_rows / len(fact), 4) if len(fact) else 0.0

findings["missing_stations"] = {
    "null_start_station_key": int(null_start_station),
    "null_end_station_key": int(null_end_station),
    "orphan_start_station_refs": int(len(orphan_start)),
    "orphan_end_station_refs": int(len(orphan_end)),
    "dim_station_missing_name": int(station_missing_name),
    "dim_station_missing_coords": int(station_missing_coords),
    "pct_rows_affected": pct_missing_station,
    "verdict": verdict(pct_missing_station),
}

print(f"NULL start_station_key : {null_start_station:,} ({null_start_station/len(fact):.2%})")
print(f"NULL end_station_key   : {null_end_station:,} ({null_end_station/len(fact):.2%})")
print(f"Orphan start_station_key refs (FK integrity): {len(orphan_start):,}")
print(f"Orphan end_station_key refs (FK integrity)  : {len(orphan_end):,}")
print(f"dim_station rows missing a name  : {station_missing_name:,}")
print(f"dim_station rows missing lat/lng : {station_missing_coords:,}")


NULL start_station_key : 1,502,107 (18.78%)
NULL end_station_key   : 1,550,226 (19.38%)
Orphan start_station_key refs (FK integrity): 0
Orphan end_station_key refs (FK integrity)  : 0
dim_station rows missing a name  : 0
dim_station rows missing lat/lng : 0


#### Invalid Timestamps

Checks every trip's `started_at`/`ended_at` pair for three kinds of problems: missing timestamps, a
trip that appears to end before (or at exactly) the moment it started — which the
`chk_fact_trip_start_before_end` database constraint should already prevent, so this re-confirms it —
and timestamps that fall outside Divvy's actual operating history: before the program's real launch
date, or somehow in the future, either of which would point to a parsing or data-entry error rather
than a real trip.

All three sub-checks are combined into one total count and one "% of rows affected" verdict, but each
is also reported individually so the root cause is visible at a glance rather than hidden inside a
single number.

**Skills demonstrated:** temporal/business-logic validation, cross-checking a database constraint from
the application layer, domain-aware range checking using the program's real launch date rather than an
arbitrary cutoff.

In [8]:
null_started = fact["started_at"].isna().sum()
null_ended = fact["ended_at"].isna().sum()

both_present = fact["started_at"].notna() & fact["ended_at"].notna()
start_ge_end = int((fact.loc[both_present, "started_at"] >= fact.loc[both_present, "ended_at"]).sum())

now_utc = pd.Timestamp.now(tz="UTC")
present_started = fact.loc[fact["started_at"].notna(), "started_at"]
out_of_program_range = int(((present_started < PROGRAM_LAUNCH) | (present_started > now_utc)).sum())

n_invalid_ts = int(null_started + null_ended + start_ge_end + out_of_program_range)
pct_invalid_ts = round(100 * n_invalid_ts / len(fact), 4) if len(fact) else 0.0

findings["invalid_timestamps"] = {
    "null_started_at": int(null_started),
    "null_ended_at": int(null_ended),
    "started_at_gte_ended_at": start_ge_end,
    "outside_program_date_range": out_of_program_range,
    "total_invalid_timestamp_rows": n_invalid_ts,
    "pct_rows_affected": pct_invalid_ts,
    "verdict": verdict(pct_invalid_ts),
}

print(f"NULL started_at              : {null_started:,}")
print(f"NULL ended_at                : {null_ended:,}")
print(f"started_at >= ended_at       : {start_ge_end:,}")
print(f"Before launch / in the future: {out_of_program_range:,}")
print(f"Total invalid-timestamp rows : {n_invalid_ts:,} ({pct_invalid_ts}% of fact_trip)")


NULL started_at              : 0
NULL ended_at                : 0
started_at >= ended_at       : 0
Before launch / in the future: 0
Total invalid-timestamp rows : 0 (0.0% of fact_trip)


#### Negative Ride Durations

Two independent checks on trip duration. First, it looks for any stored `duration_minutes` that's
zero or negative — something the `chk_fact_trip_duration_positive` database constraint should already
reject, so finding any here would point to a constraint bypass. Second, and more subtly, it recomputes
duration directly from the raw `ended_at - started_at` timestamps and compares that to the stored
value, flagging any row where the two disagree by more than half a minute. That second check is the
one that actually catches silent bugs — a `duration_minutes` value can pass the "is it positive" test
and still be wrong if it was calculated incorrectly upstream.

**Skills demonstrated:** cross-validating a derived/engineered field against its raw source
columns — a technique for catching silent calculation bugs that a simple range check alone would
miss.

In [9]:
non_positive_duration = int((fact["duration_minutes"] <= 0).sum())

computed_duration = (fact["ended_at"] - fact["started_at"]).dt.total_seconds() / 60.0
duration_mismatch = both_present & ((computed_duration - fact["duration_minutes"]).abs() > DURATION_TOLERANCE_MIN)
n_duration_mismatch = int(duration_mismatch.sum())

pct_bad_duration = round(100 * (non_positive_duration + n_duration_mismatch) / len(fact), 4) if len(fact) else 0.0

findings["negative_durations"] = {
    "non_positive_duration_rows": non_positive_duration,
    "duration_mismatch_vs_recomputed": n_duration_mismatch,
    "tolerance_minutes": DURATION_TOLERANCE_MIN,
    "pct_rows_affected": pct_bad_duration,
    "verdict": verdict(pct_bad_duration),
}

print(f"duration_minutes <= 0                         : {non_positive_duration:,}")
print(f"duration_minutes vs recomputed mismatch (>{DURATION_TOLERANCE_MIN}min): {n_duration_mismatch:,}")
if non_positive_duration:
    display(fact.loc[fact['duration_minutes'] <= 0, ['ride_id', 'started_at', 'ended_at', 'duration_minutes']].head(10))


duration_minutes <= 0                         : 0
duration_minutes vs recomputed mismatch (>0.5min): 171


#### Impossible Coordinates

Deliberately separates two things that look similar but mean very different things for data quality:
coordinates that are physically **impossible** anywhere on Earth (outside valid latitude/longitude
ranges, or the classic `(0, 0)` "null island" placeholder that shows up when geocoding silently fails),
versus coordinates that are perfectly valid but simply describe a trip that started or ended
**outside the Chicago service area** — a legitimate outlier worth flagging, not a data-quality error.
A small helper function, `_out_of_range()`, performs the range check once and is reused for both the
"impossible anywhere" test and the "outside Chicago" test, simply by passing in a different bounding
box.

**Skills demonstrated:** distinguishing genuine data errors from valid-but-unusual outliers,
reusable/parameterized validation logic, domain-aware geospatial checks.

In [10]:
def _out_of_range(lat_col, lng_col, lat_range, lng_range):
    lat_bad = fact[lat_col].notna() & ~fact[lat_col].between(*lat_range)
    lng_bad = fact[lng_col].notna() & ~fact[lng_col].between(*lng_range)
    return lat_bad | lng_bad

impossible_start = _out_of_range("start_lat", "start_lng", WORLD_LAT_RANGE, WORLD_LNG_RANGE)
impossible_end = _out_of_range("end_lat", "end_lng", WORLD_LAT_RANGE, WORLD_LNG_RANGE)
n_impossible = int((impossible_start | impossible_end).sum())

null_island = (
    ((fact["start_lat"] == 0) & (fact["start_lng"] == 0))
    | ((fact["end_lat"] == 0) & (fact["end_lng"] == 0))
)
n_null_island = int(null_island.sum())

out_of_service_area = _out_of_range("start_lat", "start_lng", CHI_LAT_RANGE, CHI_LNG_RANGE) & ~impossible_start
n_out_of_service_area = int(out_of_service_area.sum())

null_coords = fact[["start_lat", "start_lng", "end_lat", "end_lng"]].isna().any(axis=1).sum()

pct_impossible = round(100 * (n_impossible + n_null_island) / len(fact), 4) if len(fact) else 0.0

findings["impossible_coordinates"] = {
    "impossible_lat_lng_rows": n_impossible,
    "null_island_0_0_rows": n_null_island,
    "out_of_chicago_service_area_rows": n_out_of_service_area,
    "any_null_coordinate_rows": int(null_coords),
    "pct_rows_affected": pct_impossible,
    "verdict": verdict(pct_impossible),
}

print(f"Physically impossible lat/lng      : {n_impossible:,}")
print(f"(0, 0) 'null island' coordinates    : {n_null_island:,}")
print(f"Valid but outside Chicago metro area: {n_out_of_service_area:,} (flagged, not an error)")
print(f"Rows with any NULL coordinate       : {null_coords:,}")


Physically impossible lat/lng      : 0
(0, 0) 'null island' coordinates    : 0
Valid but outside Chicago metro area: 0 (flagged, not an error)
Rows with any NULL coordinate       : 9,379


#### Null Percentages

Steps back from any single column and instead runs the same null-rate calculation across **every
column, in every table** in the schema — fact and dimension tables alike. This is the raw data behind
the missing-value heatmap built later in the notebook, and it also surfaces, in one place, any column
whose null rate exceeds 5% — a useful early-warning threshold for "this field may not be reliable
enough to build an analysis on."

**Skills demonstrated:** systematic, schema-wide data profiling rather than checking columns one at a
time as an afterthought.

In [11]:
null_summary_rows = []
for tname, tdf in tables.items():
    for col in tdf.columns:
        n_null = int(tdf[col].isna().sum())
        null_summary_rows.append({
            "table": tname,
            "column": col,
            "n_rows": len(tdf),
            "n_null": n_null,
            "null_pct": round(100 * n_null / len(tdf), 3) if len(tdf) else 0.0,
        })

null_summary = pd.DataFrame(null_summary_rows).sort_values(["table", "null_pct"], ascending=[True, False])
findings["null_percentages"] = {
    "columns_over_5pct_null": null_summary.loc[null_summary["null_pct"] > 5, ["table", "column", "null_pct"]].to_dict("records"),
    "max_null_pct": float(null_summary["null_pct"].max()) if not null_summary.empty else 0.0,
}

display(null_summary[null_summary["null_pct"] > 0].reset_index(drop=True))


,table,column,n_rows,n_null,null_pct
0,fact_trip,is_round_trip,8000000,2325538,29.069
1,fact_trip,end_station_key,8000000,1550226,19.378
2,fact_trip,start_station_key,8000000,1502107,18.776
3,fact_trip,end_lat,8000000,9379,0.117
4,fact_trip,end_lng,8000000,9379,0.117


#### Cardinality

Counts the number of distinct values in each of the fact table's key columns, and expresses that as a
*uniqueness ratio* (`n_unique / n_rows`). That single ratio tells a quick, useful story about each
column's role: a ratio near 1 means the column behaves like an identifier (as expected for
`ride_id`), a ratio near 0 means it behaves like a low-cardinality category (as expected for
`day_of_week` or `member_type_key`), and anything unexpectedly in between is worth a closer look. The
same idea is applied to each dimension table's own natural key, confirming the dimensions themselves
aren't quietly carrying duplicate "different key, same real-world entity" rows.

**Skills demonstrated:** using distinct-value ratios as a fast diagnostic for whether a column is
behaving the way its role in the schema assumes it should.

In [12]:
cardinality_cols = [
    "ride_id", "start_station_key", "end_station_key", "ride_type_key",
    "member_type_key", "start_hour", "day_of_week", "month_partition",
]
cardinality = pd.DataFrame({
    "n_unique": fact[cardinality_cols].nunique(),
    "n_rows": len(fact),
})
cardinality["uniqueness_ratio"] = (cardinality["n_unique"] / cardinality["n_rows"]).round(4)

dim_cardinality = pd.DataFrame({
    "dim_station.station_id": [dim_station["station_id"].nunique()],
    "dim_ride_type.rideable_type": [dim_ride_type["rideable_type"].nunique()],
    "dim_member_type.member_casual": [dim_member_type["member_casual"].nunique()],
}).T.rename(columns={0: "n_unique"})

findings["cardinality"] = {
    "fact_trip": cardinality.to_dict("index"),
    "dimensions": dim_cardinality["n_unique"].to_dict(),
}

display(cardinality)
display(dim_cardinality)


,n_unique,n_rows,uniqueness_ratio
ride_id,8000000,8000000,1.0000
start_station_key,3215,8000000,0.0004
end_station_key,3205,8000000,0.0004
ride_type_key,3,8000000,0.0000
member_type_key,2,8000000,0.0000
start_hour,24,8000000,0.0000
day_of_week,7,8000000,0.0000
month_partition,18,8000000,0.0000


,n_unique
dim_station.station_id,3943
dim_ride_type.rideable_type,3
dim_member_type.member_casual,2


#### Data Completeness

Two different notions of "complete" are checked here. First, column-level completeness on the handful
of fields nearly every downstream analysis depends on (`ride_id`, both timestamps, both station keys,
`duration_minutes`), averaged into one overall completeness score. Second, a *calendar-coverage*
check: does `dim_date` actually contain every single day between the earliest and latest date in
range (a gap there would mean date-based joins silently drop rows), and are there any days within that
range with zero recorded rides — which could reflect a genuine service disruption, or could be a sign
that an entire month's data never made it through the ETL pipeline?

**Skills demonstrated:** measuring completeness at both the column level and the time-series/calendar
level — the second of which is easy to overlook but catches a very different class of gap than a
simple null-count.

In [13]:
critical_cols = ["ride_id", "started_at", "ended_at", "start_station_key", "end_station_key", "duration_minutes"]
completeness_by_col = (1 - fact[critical_cols].isna().mean()) * 100
overall_completeness = round(completeness_by_col.mean(), 2)

full_calendar = pd.date_range(dim_date["full_date"].min(), dim_date["full_date"].max(), freq="D")
dim_dates_present = set(pd.to_datetime(dim_date["full_date"]).dt.date)
missing_calendar_days = sorted(set(full_calendar.date) - dim_dates_present)

ride_days = pd.to_datetime(fact["started_at"], utc=True).dt.date
days_with_no_rides = sorted(set(full_calendar.date) - set(ride_days.dropna()))

findings["data_completeness"] = {
    "completeness_by_column_pct": completeness_by_col.round(2).to_dict(),
    "overall_completeness_score": overall_completeness,
    "missing_calendar_days_in_dim_date": len(missing_calendar_days),
    "days_with_zero_rides": len(days_with_no_rides),
    "date_range": [str(full_calendar.min().date()), str(full_calendar.max().date())],
}

print("Completeness by critical column:")
print(completeness_by_col.round(2))
print(f"\nOverall completeness score: {overall_completeness}%")
print(f"Missing calendar days in dim_date : {len(missing_calendar_days)} (of {len(full_calendar)} days in range)")
print(f"Days with zero recorded rides      : {len(days_with_no_rides)}")


Completeness by critical column:
ride_id              100.00
started_at           100.00
ended_at             100.00
start_station_key     81.22
end_station_key       80.62
duration_minutes     100.00
dtype: float64

Overall completeness score: 93.64%
Missing calendar days in dim_date : 0 (of 974 days in range)
Days with zero recorded rides      : 426


#### Data Consistency

The most comprehensive check in the notebook. It re-verifies referential integrity against **every**
dimension table (dates, ride type, member type) by looking for foreign keys on the fact table that
don't resolve to any real dimension row, then goes a step further and checks whether four *derived*
columns still agree with the raw fields they were computed from — `day_of_week` and `month_partition`
against `started_at`, `start_date_key` against `started_at`, and `is_round_trip` against whether the
start and end station keys actually match. Any disagreement between a derived column and the raw data
it should be computed from is a strong signal of a bug in the ETL logic
(`03_data_cleaning_and_loading.ipynb`) rather than a data-entry problem, since these fields are all
supposed to be deterministic functions of other columns already present in the same row.

**Skills demonstrated:** cross-table referential-integrity testing, validating derived/engineered
features against their source data — the same category of check that catches feature-engineering bugs
before they ever reach a machine learning model.

In [ ]:
valid_date_keys = set(dim_date["date_key"])
valid_ride_type_keys = set(dim_ride_type["ride_type_key"])
valid_member_type_keys = set(dim_member_type["member_type_key"])

orphan_start_date = fact.loc[fact["start_date_key"].notna() & ~fact["start_date_key"].isin(valid_date_keys)]
orphan_end_date = fact.loc[fact["end_date_key"].notna() & ~fact["end_date_key"].isin(valid_date_keys)]
orphan_ride_type = fact.loc[fact["ride_type_key"].notna() & ~fact["ride_type_key"].isin(valid_ride_type_keys)]
orphan_member_type = fact.loc[fact["member_type_key"].notna() & ~fact["member_type_key"].isin(valid_member_type_keys)]

sub = fact.loc[fact["started_at"].notna() & fact["day_of_week"].notna()].copy()
expected_dow = sub["started_at"].dt.dayofweek + 1
dow_mismatch = int((sub["day_of_week"] != expected_dow).sum())

sub_mp = fact.loc[fact["started_at"].notna() & fact["month_partition"].notna()].copy()
expected_mp = sub_mp["started_at"].dt.year * 100 + sub_mp["started_at"].dt.month
mp_mismatch = int((sub_mp["month_partition"] != expected_mp).sum())

sub_sdk = fact.loc[fact["started_at"].notna() & fact["start_date_key"].notna()].copy()
expected_sdk = sub_sdk["started_at"].dt.strftime("%Y%m%d").astype("int64")
sdk_mismatch = int((sub_sdk["start_date_key"] != expected_sdk).sum())

has_both_stations = fact["start_station_key"].notna() & fact["end_station_key"].notna() & fact["is_round_trip"].notna()
expected_round_trip = fact.loc[has_both_stations, "start_station_key"] == fact.loc[has_both_stations, "end_station_key"]
round_trip_mismatch = int((fact.loc[has_both_stations, "is_round_trip"] != expected_round_trip).sum())

n_consistency_issues = int(
    len(orphan_start_date) + len(orphan_end_date) + len(orphan_ride_type) + len(orphan_member_type)
    + dow_mismatch + mp_mismatch + sdk_mismatch + round_trip_mismatch
)
pct_consistency_issues = round(100 * n_consistency_issues / len(fact), 4) if len(fact) else 0.0

findings["data_consistency"] = {
    "orphan_start_date_refs": int(len(orphan_start_date)),
    "orphan_end_date_refs": int(len(orphan_end_date)),
    "orphan_ride_type_refs": int(len(orphan_ride_type)),
    "orphan_member_type_refs": int(len(orphan_member_type)),
    "day_of_week_mismatches": dow_mismatch,
    "month_partition_mismatches": mp_mismatch,
    "start_date_key_mismatches": sdk_mismatch,
    "is_round_trip_mismatches": round_trip_mismatch,
    "total_consistency_issues": n_consistency_issues,
    "pct_rows_affected": pct_consistency_issues,
    "verdict": verdict(pct_consistency_issues),
}

print(f"Orphan start_date_key / end_date_key refs   : {len(orphan_start_date):,} / {len(orphan_end_date):,}")
print(f"Orphan ride_type_key / member_type_key refs : {len(orphan_ride_type):,} / {len(orphan_member_type):,}")
print(f"day_of_week mismatches vs started_at        : {dow_mismatch:,}")
print(f"month_partition mismatches vs started_at     : {mp_mismatch:,}")
print(f"start_date_key mismatches vs started_at      : {sdk_mismatch:,}")
print(f"is_round_trip mismatches vs station keys     : {round_trip_mismatch:,}")
print(f"Total consistency issues                     : {n_consistency_issues:,} ({pct_consistency_issues}% of fact_trip)")


### Missing Value Matrix

Turns the null-percentage numbers from the previous section into two complementary pictures, because
a table of numbers is much harder to scan for patterns than a chart:

1. A **table x column heatmap** of null percentages — makes it immediately obvious, at a glance, which
   tables and columns across the *entire* schema need attention, without reading through dozens of
   rows of numbers.
2. A **row-level presence matrix** for a sample of `fact_trip` — the classic black-is-present /
   white-is-missing grid popularized by the `missingno` library — which answers a different question:
   do missing values happen independently, or do they *cluster* (e.g. rows missing
   `end_station_key` also tend to be missing `is_round_trip`, which would suggest a shared root cause
   rather than two unrelated problems)?

The heatmap (built first) colors every table/column combination by its null percentage and annotates
each cell with the exact number, so it works as both a quick visual scan and a precise lookup. The
row-level matrix (built second) samples up to 500 rows for readability and renders every remaining
column as a black/white presence grid, saving both charts to disk alongside the rest of the report.

**Skills demonstrated:** data visualization for exploratory data-quality analysis, choosing the right
chart for the question being asked (aggregate heatmap vs. row-level pattern detection), reusable chart
styling and export for a written report.

In [ ]:
pivot = null_summary.pivot(index="column", columns="table", values="null_pct")

fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(pivot))))
im = ax.imshow(pivot.fillna(-1).values, cmap="Reds", vmin=0, vmax=max(5, pivot.max().max()), aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                     fontsize=8, color="white" if val > pivot.max().max()/2 else "black")
ax.set_title("Null Percentage by Table x Column")
fig.colorbar(im, ax=ax, label="% null")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_FOLDER, "missing_value_heatmap.png"), dpi=150)
plt.show()


In [ ]:
sample_n = min(500, len(fact))
sample_cols = [c for c in fact.columns if c not in ("trip_key",)]
sample = fact[sample_cols].sample(n=sample_n, random_state=42).reset_index(drop=True) if sample_n else fact[sample_cols]
presence = sample.notna().astype(int)

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(presence.values, cmap="Greys", aspect="auto", interpolation="nearest")
ax.set_xticks(range(len(presence.columns)))
ax.set_xticklabels(presence.columns, rotation=75, ha="right", fontsize=8)
ax.set_yticks([])
ax.set_ylabel(f"Sampled rows (n={sample_n})")
ax.set_title("Row-Level Missing Value Matrix — fact_trip sample (black = present, white = missing)")
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_FOLDER, "missing_value_matrix_rows.png"), dpi=150)
plt.show()


### Composite Data Quality Score

Nine separate checks produce nine separate numbers — useful for debugging, but not something a hiring
manager, stakeholder, or future version of this notebook can track at a glance. This section solves
that by rolling every check up into five industry-standard data-quality dimensions, each scored 0–100
as `100 - % of rows affected` (floored at 0), then combining those five dimension scores into one
weighted overall score with a letter grade:

| Dimension    | Weight | Driven by |
|---|---|---|
| Uniqueness   | 20% | duplicate `ride_id` rate |
| Completeness | 20% | critical-column null rate |
| Validity     | 25% | invalid timestamps + non-positive/mismatched durations + impossible coordinates |
| Consistency  | 20% | orphaned foreign keys + derived-field mismatches |
| Accuracy     | 15% | `is_anomalous` flag rate + calendar coverage gaps |

Giving Validity the heaviest weight (25%) reflects a deliberate judgment call: a warehouse can tolerate
a *few* missing values or minor inconsistencies far more easily than it can tolerate physically invalid
data — impossible coordinates, negative durations — making it into an analysis. The result, a single
`overall_score` and letter grade, is exactly the kind of headline KPI a data-quality dashboard or
executive summary needs, computed transparently from checks that are fully documented above rather
than an opaque black-box score.

A second short cell then turns those five dimension scores into a simple color-coded bar chart (green
at 90+, amber at 70+, red below), so the score is just as easy to present visually as it is to read as
a number.

**Skills demonstrated:** composite KPI/scorecard design, weighted multi-dimension scoring, translating
granular technical checks into an executive-readable metric, data visualization for stakeholder
communication.

In [ ]:
def score_from_pct(pct_affected: float) -> float:
    return max(0.0, 100.0 - pct_affected)

n = len(fact)

uniqueness_score = score_from_pct(findings["duplicate_ride_ids"]["pct_rows_affected"])
completeness_score = findings["data_completeness"]["overall_completeness_score"]

validity_bad_rows = (
    findings["invalid_timestamps"]["total_invalid_timestamp_rows"]
    + findings["negative_durations"]["non_positive_duration_rows"]
    + findings["negative_durations"]["duration_mismatch_vs_recomputed"]
    + findings["impossible_coordinates"]["impossible_lat_lng_rows"]
    + findings["impossible_coordinates"]["null_island_0_0_rows"]
)
validity_pct = round(100 * validity_bad_rows / n, 4) if n else 0.0
validity_score = score_from_pct(validity_pct)

consistency_score = score_from_pct(findings["data_consistency"]["pct_rows_affected"])

anomalous_rate_pct = round(100 * fact["is_anomalous"].fillna(False).mean(), 4) if "is_anomalous" in fact else 0.0
calendar_gap_pct = round(
    100 * findings["data_completeness"]["missing_calendar_days_in_dim_date"]
    / max(1, (pd.to_datetime(findings["data_completeness"]["date_range"][1])
              - pd.to_datetime(findings["data_completeness"]["date_range"][0])).days + 1),
    4,
)
accuracy_score = score_from_pct(anomalous_rate_pct + calendar_gap_pct)

dimension_scores = {
    "Uniqueness": round(uniqueness_score, 2),
    "Completeness": round(completeness_score, 2),
    "Validity": round(validity_score, 2),
    "Consistency": round(consistency_score, 2),
    "Accuracy": round(accuracy_score, 2),
}
weights = {"Uniqueness": 0.20, "Completeness": 0.20, "Validity": 0.25, "Consistency": 0.20, "Accuracy": 0.15}

overall_score = round(sum(dimension_scores[k] * weights[k] for k in weights), 2)

def grade_for(score):
    if score >= 90: return "A"
    if score >= 80: return "B"
    if score >= 70: return "C"
    if score >= 60: return "D"
    return "F"

overall_grade = grade_for(overall_score)

findings["quality_score"] = {
    "dimension_scores": dimension_scores,
    "weights": weights,
    "overall_score": overall_score,
    "overall_grade": overall_grade,
}

score_df = pd.DataFrame({
    "dimension": list(dimension_scores.keys()),
    "score": list(dimension_scores.values()),
    "weight": [weights[k] for k in dimension_scores],
})
display(score_df)
print(f"\nOVERALL DATA QUALITY SCORE: {overall_score} / 100  (Grade {overall_grade})")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#2e7d32" if s >= 90 else "#f9a825" if s >= 70 else "#c62828" for s in dimension_scores.values()]
bars = ax.bar(dimension_scores.keys(), dimension_scores.values(), color=colors)
ax.set_ylim(0, 100)
ax.set_ylabel("Score (0-100)")
ax.set_title(f"Data Quality Dimension Scores — Overall: {overall_score} (Grade {overall_grade})")
ax.axhline(90, color="grey", linestyle="--", linewidth=0.8)
for b, s in zip(bars, dimension_scores.values()):
    ax.text(b.get_x() + b.get_width()/2, s + 1.5, f"{s:.1f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(REPORTS_FOLDER, "quality_score_breakdown.png"), dpi=150)
plt.show()


### Data Quality Report

Everything computed across the entire notebook — every check's findings, the null summary, the
cardinality table, the completeness stats, and the final quality score — gets compiled here into one
structured report and rendered directly in the notebook as formatted Markdown, so a reviewer can see
the full picture without re-running anything. A dedicated `build_markdown_report()` function assembles
the report section by section (run metadata, table row counts, a summary table of every check's
verdict, the null-percentage table, cardinality, completeness, and the final score breakdown), keeping
the report-generation logic cleanly separate from the print statements used everywhere else in the
notebook.

The second cell then persists that same set of findings to disk in three complementary formats: a
human-readable Markdown report for a person to read, a full-detail JSON file for another script or
pipeline to consume programmatically, and a flattened CSV of every individual metric for a spreadsheet
or BI tool like Power BI to import directly. Producing all three from one shared `findings` dictionary
means the numbers in every format are guaranteed to match — there's no risk of the Markdown report and
the CSV export drifting out of sync.

**Skills demonstrated:** end-to-end automated reporting, multi-format data deliverables (Markdown/
JSON/CSV) built for different audiences, separating computation from presentation, single-source-of-
truth reporting design.

In [ ]:
def build_markdown_report() -> str:
    lines = []
    lines.append(f"# Divvy Data Quality Report")
    lines.append(f"**Run timestamp (UTC):** {RUN_TIMESTAMP}  ")
    lines.append(f"**Rows assessed (fact_trip):** {len(fact):,}  ")
    lines.append(f"**Overall Quality Score:** {overall_score} / 100 (Grade {overall_grade})\n")

    lines.append("## Table Row Counts")
    for tname, tdf in tables.items():
        lines.append(f"- `{tname}`: {len(tdf):,} rows")
    lines.append("")

    lines.append("## Check Results\n")
    lines.append("| # | Check | Key Metric | % Rows Affected | Verdict |")
    lines.append("|---|---|---|---|---|")
    duplicate_ids = findings["duplicate_ride_ids"]

    duplicate_ids = findings["duplicate_ride_ids"]
    stations = findings["missing_stations"]
    timestamps = findings["invalid_timestamps"]
    durations = findings["negative_durations"]
    coordinates = findings["impossible_coordinates"]
    consistency = findings["data_consistency"]

    check_rows = [
    	(
            1,
            "Duplicate ride IDs",
            f"{duplicate_ids['duplicate_rows_total']:,} duplicate rows",
            duplicate_ids["pct_rows_affected"],
            duplicate_ids["verdict"],
        ),
        (
            2,
            "Missing stations",
            (
                f"{stations['null_start_station_key'] + stations['null_end_station_key']:,} "
                "null station refs"
            ),
            stations["pct_rows_affected"],
            stations["verdict"],
        ),
        (
            3,
            "Invalid timestamps",
            f"{timestamps['total_invalid_timestamp_rows']:,} invalid rows",
            timestamps["pct_rows_affected"],
            timestamps["verdict"],
        ),
        (
            4,
            "Negative ride durations",
            (
                f"{durations['non_positive_duration_rows']:,} non-positive + "
                f"{durations['duration_mismatch_vs_recomputed']:,} mismatched"
            ),
            durations["pct_rows_affected"],
            durations["verdict"],
        ),
        (
            5,
            "Impossible coordinates",
            (
                f"{coordinates['impossible_lat_lng_rows']:,} impossible + "
                f"{coordinates['null_island_0_0_rows']:,} (0,0)"
            ),
            coordinates["pct_rows_affected"],
            coordinates["verdict"],
        ),
        (
            9,
            "Data consistency",
            f"{consistency['total_consistency_issues']:,} issues",
            consistency["pct_rows_affected"],
            consistency["verdict"],
        ),
    ]

    for n_, check, metric, pct, verd in check_rows:
        lines.append(f"| {n_} | {check} | {metric} | {pct}% | {verd} |")
    lines.append("")

    lines.append("## Null Percentages (columns with any nulls)\n")
    lines.append(null_summary[null_summary['null_pct'] > 0].to_markdown(index=False))
    lines.append("")

    lines.append("## Cardinality (fact_trip)\n")
    lines.append(cardinality.to_markdown())
    lines.append("")

    lines.append("## Data Completeness\n")
    lines.append(f"- Overall completeness score: **{findings['data_completeness']['overall_completeness_score']}%**")
    lines.append(f"- Missing calendar days in `dim_date`: {findings['data_completeness']['missing_calendar_days_in_dim_date']}")
    lines.append(f"- Days with zero recorded rides: {findings['data_completeness']['days_with_zero_rides']}")
    lines.append(f"- Date range assessed: {findings['data_completeness']['date_range'][0]} to {findings['data_completeness']['date_range'][1]}")
    lines.append("")

    lines.append("## Data Quality Score Breakdown\n")
    lines.append(score_df.to_markdown(index=False))
    lines.append("")
    lines.append(f"**Overall: {overall_score} / 100 — Grade {overall_grade}**")

    return "\n".join(lines)


report_md = build_markdown_report()
display(Markdown(report_md))


In [ ]:
# Persist all three report formats
md_path = os.path.join(REPORTS_FOLDER, "data_quality_report.md")
json_path = os.path.join(REPORTS_FOLDER, "data_quality_report.json")
csv_path = os.path.join(REPORTS_FOLDER, "data_quality_metrics.csv")

with open(md_path, "w", encoding="utf-8") as f:
    f.write(report_md)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(findings, f, indent=2, default=str)

# Flatten findings into one metrics-per-row CSV for spreadsheet/BI consumption
flat_rows = []
for check, metrics in findings.items():
    if not isinstance(metrics, dict):
        continue
    for k, v in metrics.items():
        if isinstance(v, (dict, list)):
            continue
        flat_rows.append({"check": check, "metric": k, "value": v})
pd.DataFrame(flat_rows).to_csv(csv_path, index=False)

print(f"Markdown report : {md_path}")
print(f"JSON report     : {json_path}")
print(f"CSV metrics     : {csv_path}")
print(f"Charts          : {REPORTS_FOLDER}")


### Summary

This assessment re-derives data-quality metrics straight from the warehouse rather than trusting the
ETL pipeline's self-reported counters, so it will also surface issues introduced after
loading — a manual edit, a partial re-run, or a constraint quietly bypassed during a bulk load.
Re-running this notebook after every load (or on a schedule) turns the **Quality Score** into a trend
line rather than a one-time snapshot: a sustained drop is an early warning that something
upstream — Divvy's export format changing, a regression in the ETL logic in
`03_etl_pipeline.ipynb`, or a manual edit to the warehouse — needs attention before it
ever reaches a dashboard or a model.

**Why this matters (for recruiters/reviewers):** this closing section is the difference between "I ran
some checks" and "I built a repeatable quality-monitoring process." Treating data quality as a
trackable score, tied back to specific, documented checks, is exactly the discipline expected of
anyone responsible for the data feeding a production dashboard or a machine learning pipeline.

**Skills demonstrated:** data-quality monitoring as an ongoing process rather than a one-off audit,
root-cause framing that ties a quality regression back to a specific upstream stage of the pipeline.